In [ ]:
!pip -q install -U "transformers>=4.41.0" "accelerate" "peft" "trl==0.9.3" "datasets" "huggingface_hub" "tqdm"

In [ ]:
#!/usr/bin/env python
"""
Goal: Train a reward model ONLY on the 1k HH-RLHF pairs (no synthetic sampling)

Pushes to HF Hub:
- repo: xxxxxxxx

Requirements:
  pip install -q torch transformers datasets accelerate huggingface_hub pandas requests matplotlib
"""

import os
import re
import time
import random
import shutil
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from huggingface_hub import login
from huggingface_hub.utils import configure_http_backend

from google.colab import drive


# =========================
# CONFIG
# =========================
drive.mount("/content/drive")
# =========================
# CONFIG
# =========================
HF_TOKEN = "xxxxxxxx"
HF_REPO_ID = "xxxxxxxx"  # <-- repo to push final model

BASE_MODEL = "microsoft/deberta-v3-base"
MAX_LENGTH = 512

BASE_CSV = "xxxxxxxx location to the 1k samples saved locally xxxxxxxx"
NUM_BASE = 1000

# Train for exactly T epochs
T = 5

# Training hyperparams
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 16
LR = 2e-5
WARMUP_RATIO = 0.0
WEIGHT_DECAY = 0.0
LOGGING_STEPS = 5

# Save metrics/plots
RUN_ROOT = "xxxxxxxx Root Directory Name xxxxxxxx"
METRICS_CSV = os.path.join(RUN_ROOT, "train_metrics.csv")
LOSS_PNG = os.path.join(RUN_ROOT, "train_loss_over_epochs.png")
ACC_PNG = os.path.join(RUN_ROOT, "train_pairwise_acc.png")
MARGIN_PNG = os.path.join(RUN_ROOT, "train_mean_margin.png")
os.makedirs(RUN_ROOT, exist_ok=True)

GLOBAL_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# HF timeouts
os.environ.setdefault("HF_HUB_HTTP_TIMEOUT", "180")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "180")

def _hf_backend_factory() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=8,
        connect=8,
        read=8,
        status=8,
        backoff_factor=1.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["HEAD", "GET", "POST", "PUT", "DELETE", "PATCH"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=16, pool_maxsize=16)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

configure_http_backend(backend_factory=_hf_backend_factory)


def safe_push_to_hub(model, tokenizer, repo_id: str, token: str, commit_message: str, max_retries: int = 6):
    """Final push only (called once). Retries on transient failures."""
    if not token or not repo_id:
        return
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            print(f"[HF] Push attempt {attempt}/{max_retries}: {repo_id} | {commit_message}")
            model.push_to_hub(
                repo_id,
                token=token,
                commit_message=commit_message,
                max_shard_size="200MB",
            )
            tokenizer.push_to_hub(
                repo_id,
                token=token,
                commit_message=commit_message,
            )
            print("[HF] Push complete.")
            return
        except Exception as e:
            last_err = e
            wait = min(60, 2 ** attempt)
            print(f"[HF] Push failed ({type(e).__name__}: {e}). Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"HF push failed after {max_retries} attempts. Last error: {last_err}")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)

_HUMAN_RE = re.compile(r"^\s*(human|user)\s*:", re.IGNORECASE)

def looks_like_full_dialogue(text: str) -> bool:
    if text is None:
        return False
    t = str(text).strip()
    if len(t) < 20:
        return False
    return bool(_HUMAN_RE.search(t)) or ("Human:" in t[:200]) or ("Assistant:" in t[:200])

def build_pair_text(prompt: str, response: str) -> str:
    r = "" if response is None else str(response)
    if looks_like_full_dialogue(r):
        return r
    p = "" if prompt is None else str(prompt)
    return f"Human: {p}\n\nAssistant: {r}"

# Load base
def load_base_df():
    if not os.path.exists(BASE_CSV):
        raise FileNotFoundError(f"Missing BASE_CSV: {BASE_CSV}")
    df = pd.read_csv(BASE_CSV)

    need = {"chosen", "rejected"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"BASE_CSV missing columns {miss}. Found {list(df.columns)}")

    if len(df) != NUM_BASE:
        print(f"[Warn] BASE_CSV has {len(df)} rows (expected {NUM_BASE}). Using all rows.")
    return df.reset_index(drop=True)

base_df = load_base_df()
base_has_prompt = "prompt" in base_df.columns
print(f"Base pairs: {len(base_df)} | base_has_prompt={base_has_prompt}")


# Model and tokenizer
def build_tokenizer_and_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1)
    if getattr(model, "resize_token_embeddings", None) is not None:
        model.resize_token_embeddings(len(tokenizer))

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.problem_type = "regression"
    model.config.use_cache = False
    model.to(DEVICE)
    return tokenizer, model


def df_to_prompt_conditioned_dataset(df_: pd.DataFrame, has_prompt: bool) -> Dataset:
    prompts = df_["prompt"].astype(str).tolist() if has_prompt else [""] * len(df_)
    chosen_texts = [build_pair_text(prompts[i], df_.iloc[i]["chosen"]) for i in range(len(df_))]
    rejected_texts = [build_pair_text(prompts[i], df_.iloc[i]["rejected"]) for i in range(len(df_))]
    return Dataset.from_dict({"chosen_text": chosen_texts, "rejected_text": rejected_texts})


def make_data_collator(tokenizer):
    def collate_fn(batch):
        chosen_texts = [str(ex["chosen_text"]) for ex in batch]
        rejected_texts = [str(ex["rejected_text"]) for ex in batch]

        chosen_enc = tokenizer(
            chosen_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        rejected_enc = tokenizer(
            rejected_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        )

        return {
            "chosen_input_ids": chosen_enc["input_ids"],
            "chosen_attention_mask": chosen_enc["attention_mask"],
            "rejected_input_ids": rejected_enc["input_ids"],
            "rejected_attention_mask": rejected_enc["attention_mask"],
        }
    return collate_fn


class PairwiseRewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        chosen_out = model(
            input_ids=inputs["chosen_input_ids"],
            attention_mask=inputs["chosen_attention_mask"],
        )
        rejected_out = model(
            input_ids=inputs["rejected_input_ids"],
            attention_mask=inputs["rejected_attention_mask"],
        )
        chosen_scores = chosen_out.logits.squeeze(-1)
        rejected_scores = rejected_out.logits.squeeze(-1)
        margin = chosen_scores - rejected_scores
        loss = -F.logsigmoid(margin).mean()
        if return_outputs:
            return loss, {"chosen_scores": chosen_scores, "rejected_scores": rejected_scores, "margin": margin}
        return loss


@torch.no_grad()
def compute_train_metrics(model, tokenizer, train_ds: Dataset, batch_size=8):
    model.eval()
    total, correct = 0, 0
    margins_all = []

    for start in range(0, len(train_ds), batch_size):
        end = min(start + batch_size, len(train_ds))
        batch = train_ds[start:end]
        chosen_texts = batch["chosen_text"]
        rejected_texts = batch["rejected_text"]

        c_enc = tokenizer(
            chosen_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)
        r_enc = tokenizer(
            rejected_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)

        c = model(**c_enc).logits.squeeze(-1)
        r = model(**r_enc).logits.squeeze(-1)
        m = (c - r).detach().float().cpu().numpy()

        margins_all.append(m)
        correct += int((m > 0).sum())
        total += int(m.shape[0])

    margins = np.concatenate(margins_all, axis=0)
    return {
        "train_pairwise_acc": float(correct / max(1, total)),
        "train_mean_margin": float(margins.mean()),
        "train_std_margin": float(margins.std()),
    }


def plot_loss_over_epochs_from_trainer(trainer: Trainer, T: int, out_png: str):
    log_hist = [x for x in trainer.state.log_history if isinstance(x, dict)]

    epoch_vals = []
    loss_vals = []
    for x in log_hist:
        if "loss" in x and "epoch" in x:
            try:
                epoch_vals.append(float(x["epoch"]))
                loss_vals.append(float(x["loss"]))
            except Exception:
                pass

    if len(loss_vals) == 0:
        print("[Warn] No per-step loss logs with epoch info found; skipping epoch-loss plot.")
        return None

    df_loss = pd.DataFrame({"epoch": epoch_vals, "loss": loss_vals})

    df_loss["epoch_bin"] = np.clip(np.floor(df_loss["epoch"]).astype(int) + 1, 1, T)

    epoch_loss = (
        df_loss.groupby("epoch_bin", as_index=False)["loss"]
        .mean()
        .sort_values("epoch_bin")
    )

    full_epochs = pd.DataFrame({"epoch_bin": np.arange(1, T + 1)})
    epoch_loss = full_epochs.merge(epoch_loss, on="epoch_bin", how="left")

    plt.figure()
    plt.plot(epoch_loss["epoch_bin"], epoch_loss["loss"], marker="o", linewidth=2.2)
    plt.xlabel("Epoch")
    plt.ylabel("Train loss (mean over steps in epoch)")
    plt.title("NoAug: train loss over epochs")
    plt.grid(True, linewidth=0.3)
    plt.xticks(np.arange(1, T + 1))
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()
    print(f"[OK] Saved epoch-wise loss plot → {out_png}")

    return epoch_loss


# MAIN

def main():
    if HF_TOKEN:
        login(HF_TOKEN)

    tokenizer, model = build_tokenizer_and_model()
    data_collator = make_data_collator(tokenizer)

    train_ds = df_to_prompt_conditioned_dataset(base_df, has_prompt=base_has_prompt)

    print("\nSanity check (sample 0 head):")
    print("--- chosen_text head ---")
    print(train_ds[0]["chosen_text"][:400])
    print("--- rejected_text head ---")
    print(train_ds[0]["rejected_text"][:400])

    # Metrics BEFORE training
    before = compute_train_metrics(model, tokenizer, train_ds, batch_size=8)
    print(f"[Before] acc={before['train_pairwise_acc']:.4f} | mean_margin={before['train_mean_margin']:.4f}")

    TMP_DIR = "/tmp/noaug_rm_trainer_out"
    if os.path.exists(TMP_DIR):
        shutil.rmtree(TMP_DIR, ignore_errors=True)
    os.makedirs(TMP_DIR, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=TMP_DIR,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        num_train_epochs=T,                   # Train for T epochs on base-only
        logging_steps=LOGGING_STEPS,
        logging_strategy="steps",
        report_to="none",
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        save_strategy="no",
        save_safetensors=False,
        seed=GLOBAL_SEED,
    )

    trainer = PairwiseRewardTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        data_collator=data_collator,
    )

    train_out = trainer.train()
    train_loss = float(train_out.metrics.get("train_loss", np.nan))
    print(f"[Train done] train_loss={train_loss:.6f}")

    # Metrics AFTER training
    after = compute_train_metrics(trainer.model, tokenizer, train_ds, batch_size=8)
    print(f"[After ] acc={after['train_pairwise_acc']:.4f} | mean_margin={after['train_mean_margin']:.4f}")

    # MARS-style: loss over epochs (1 point per epoch)
    epoch_loss_df = plot_loss_over_epochs_from_trainer(trainer, T=T, out_png=LOSS_PNG)

    # Save metrics CSV + plots (Drive only)
    row = {
        "epochs": T,
        "train_loss": train_loss,
        "acc_before": before["train_pairwise_acc"],
        "acc_after": after["train_pairwise_acc"],
        "mean_margin_before": before["train_mean_margin"],
        "mean_margin_after": after["train_mean_margin"],
        "std_margin_after": after["train_std_margin"],
        "lr": LR,
        "gas": GRAD_ACCUM_STEPS,
        "batch": PER_DEVICE_TRAIN_BATCH,
    }

    # If epoch-wise losses exist, store them in CSV too
    if epoch_loss_df is not None:
        for e in range(1, T + 1):
            v = epoch_loss_df.loc[epoch_loss_df["epoch_bin"] == e, "loss"].values
            row[f"epoch_{e}_mean_loss"] = float(v[0]) if len(v) else np.nan

    metrics_df = pd.DataFrame([row])
    metrics_df.to_csv(METRICS_CSV, index=False)
    print(f"[OK] Saved metrics CSV → {METRICS_CSV}")

    # Acc bar
    plt.figure()
    plt.bar(["before", "after"], [before["train_pairwise_acc"], after["train_pairwise_acc"]])
    plt.ylabel("Train pairwise accuracy")
    plt.title("NoAug fixed1k — train pairwise accuracy")
    plt.grid(True, axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(ACC_PNG, dpi=200)
    plt.close()
    print(f"[OK] Saved acc plot → {ACC_PNG}")

    # Margin bar
    plt.figure()
    plt.bar(["before", "after"], [before["train_mean_margin"], after["train_mean_margin"]])
    plt.ylabel("Train mean margin")
    plt.title("NoAug fixed1k — train mean margin")
    plt.grid(True, axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(MARGIN_PNG, dpi=200)
    plt.close()
    print(f"[OK] Saved margin plot → {MARGIN_PNG}")

    # FINAL PUSH ONLY
    final_model = trainer.model
    if HF_TOKEN and HF_REPO_ID:
        print("\nPushing FINAL model + tokenizer to Hugging Face Hub...")
        login(token=HF_TOKEN)
        safe_push_to_hub(
            model=final_model,
            tokenizer=tokenizer,
            repo_id=HF_REPO_ID,
            token=HF_TOKEN,
            commit_message=f"final: no-aug fixed1k (epochs={T}, lr={LR}, gas={GRAD_ACCUM_STEPS})",
        )
        print(f"[OK] Final push complete → {HF_REPO_ID}")
    else:
        print("\n[Info] HF_TOKEN or HF_REPO_ID missing -> skipping HF push.")

    # Cleanup temp
    del trainer
    torch.cuda.empty_cache()
    if os.path.exists(TMP_DIR):
        shutil.rmtree(TMP_DIR, ignore_errors=True)

    print("\n COMPLETED! ")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python
"""
Goal:

Uniform Augmentation rule:
- Iteration t=1..T:
    - Always include the SAME 1k base pairs
    - Sample exactly TOTAL_BUDGET synthetic pairs uniformly from synthetic pool
      (TOTAL_BUDGET must be divisible by NUM_BASE if you want SYN_PER_BASE integer;
       but this script only needs TOTAL_BUDGET total.)

Requirements:
  pip install -q torch transformers datasets accelerate huggingface_hub pandas tqdm requests matplotlib
"""

import os
import re
import time
import random
import shutil
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from huggingface_hub import login
from huggingface_hub.utils import configure_http_backend


# CONFIG
HF_TOKEN = "xxxxxxxxxx"  # <-- your real token (hf_...). Set "" to disable pushing.
HF_REPO_ID = "xxxxxxxxxx"  # <-- requested repo name

BASE_MODEL = "microsoft/deberta-v3-base"
MAX_LENGTH = 512

BASE_CSV = "xxxxxxxx location to the 1k samples saved locally xxxxxxxx"
SYNTH_CSV = "xxxxxxxx location to the pre-generated synthetic samples saved locally xxxxxxxx"

NUM_BASE = 1000

# Budget controls synthetic sampling
TOTAL_BUDGET = 2000   # number of synthetic pairs per iteration
T = 5

# Save ONLY sampled synthetic audits + metrics (NOT model)
RUN_ROOT = "xxxxxxxx location to the root directory xxxxxxxx"
SAMPLED_DIR = os.path.join(RUN_ROOT, "sampled_synth_iters")
METRICS_CSV = os.path.join(RUN_ROOT, "iter_metrics.csv")
LOSS_PNG = os.path.join(RUN_ROOT, "mean_train_loss_over_iters.png")
ACC_PNG = os.path.join(RUN_ROOT, "train_pairwise_acc_over_iters.png")
MARGIN_PNG = os.path.join(RUN_ROOT, "train_mean_margin_over_iters.png")
os.makedirs(SAMPLED_DIR, exist_ok=True)
os.makedirs(RUN_ROOT, exist_ok=True)

# Training hyperparams
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 16
LR = 2e-5
WARMUP_RATIO = 0.0  # recommend 0 for small runs debugging
WEIGHT_DECAY = 0.0
LOGGING_STEPS = 5

GLOBAL_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# HF timeouts
os.environ.setdefault("HF_HUB_HTTP_TIMEOUT", "180")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "180")


def _hf_backend_factory() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=8,
        connect=8,
        read=8,
        status=8,
        backoff_factor=1.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["HEAD", "GET", "POST", "PUT", "DELETE", "PATCH"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=16, pool_maxsize=16)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

configure_http_backend(backend_factory=_hf_backend_factory)


def safe_push_to_hub(model, tokenizer, repo_id: str, token: str, commit_message: str, max_retries: int = 6):
    """Final push only (called once at end). Retries on transient failures."""
    if not token or not repo_id:
        return
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            print(f"[HF] Push attempt {attempt}/{max_retries}: {repo_id} | {commit_message}")
            model.push_to_hub(
                repo_id,
                token=token,
                commit_message=commit_message,
                max_shard_size="200MB",
            )
            tokenizer.push_to_hub(
                repo_id,
                token=token,
                commit_message=commit_message,
            )
            print("[HF] Push complete.")
            return
        except Exception as e:
            last_err = e
            wait = min(60, 2 ** attempt)
            print(f"[HF] Push failed ({type(e).__name__}: {e}). Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"HF push failed after {max_retries} attempts. Last error: {last_err}")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)



_HUMAN_RE = re.compile(r"^\s*(human|user)\s*:", re.IGNORECASE)

def looks_like_full_dialogue(text: str) -> bool:
    if text is None:
        return False
    t = str(text).strip()
    if len(t) < 20:
        return False
    return bool(_HUMAN_RE.search(t)) or ("Human:" in t[:200]) or ("Assistant:" in t[:200])

def build_pair_text(prompt: str, response: str) -> str:
    r = "" if response is None else str(response)
    if looks_like_full_dialogue(r):
        return r
    p = "" if prompt is None else str(prompt)
    return f"Human: {p}\n\nAssistant: {r}"



# Load base and synthetic

def load_base_df():
    if not os.path.exists(BASE_CSV):
        raise FileNotFoundError(f"Missing BASE_CSV: {BASE_CSV}")
    df = pd.read_csv(BASE_CSV)

    need = {"chosen", "rejected"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"BASE_CSV missing columns {miss}. Found {list(df.columns)}")

    if len(df) != NUM_BASE:
        print(f"[Warn] BASE_CSV has {len(df)} rows (expected {NUM_BASE}). Using all rows.")
    return df.reset_index(drop=True)


def load_synth_df():
    if not os.path.exists(SYNTH_CSV):
        raise FileNotFoundError(f"Missing SYNTH_CSV: {SYNTH_CSV}")
    df = pd.read_csv(SYNTH_CSV)

    need = {"chosen", "rejected"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"SYNTH_CSV missing columns {miss}. Found {list(df.columns)}")

    # Exclude original pairs if the file includes them
    if "pair_type" in df.columns:
        df = df[df["pair_type"] != "original"].copy()
    if "pair_id" in df.columns:
        df = df[df["pair_id"] != 0].copy()

    return df.reset_index(drop=True)


base_df = load_base_df()
synth_df = load_synth_df()

base_has_prompt = "prompt" in base_df.columns
synth_has_prompt = "prompt" in synth_df.columns

print(f"Base pairs: {len(base_df)} | base_has_prompt={base_has_prompt}")
print(f"Synthetic pool: {len(synth_df)} | synth_has_prompt={synth_has_prompt}")


# Model AND tokenizer

def build_tokenizer_and_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1)

    if getattr(model, "resize_token_embeddings", None) is not None:
        model.resize_token_embeddings(len(tokenizer))

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.problem_type = "regression"
    model.config.use_cache = False
    model.to(DEVICE)
    return tokenizer, model

def df_to_prompt_conditioned_dataset(df_: pd.DataFrame, has_prompt: bool) -> Dataset:
    prompts = df_["prompt"].astype(str).tolist() if has_prompt else [""] * len(df_)
    chosen_texts = [build_pair_text(prompts[i], df_.iloc[i]["chosen"]) for i in range(len(df_))]
    rejected_texts = [build_pair_text(prompts[i], df_.iloc[i]["rejected"]) for i in range(len(df_))]
    return Dataset.from_dict({"chosen_text": chosen_texts, "rejected_text": rejected_texts})


def make_data_collator(tokenizer):
    def collate_fn(batch):
        chosen_texts = [str(ex["chosen_text"]) for ex in batch]
        rejected_texts = [str(ex["rejected_text"]) for ex in batch]

        chosen_enc = tokenizer(
            chosen_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        rejected_enc = tokenizer(
            rejected_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        )

        return {
            "chosen_input_ids": chosen_enc["input_ids"],
            "chosen_attention_mask": chosen_enc["attention_mask"],
            "rejected_input_ids": rejected_enc["input_ids"],
            "rejected_attention_mask": rejected_enc["attention_mask"],
        }
    return collate_fn


class PairwiseRewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        chosen_out = model(
            input_ids=inputs["chosen_input_ids"],
            attention_mask=inputs["chosen_attention_mask"],
        )
        rejected_out = model(
            input_ids=inputs["rejected_input_ids"],
            attention_mask=inputs["rejected_attention_mask"],
        )
        chosen_scores = chosen_out.logits.squeeze(-1)
        rejected_scores = rejected_out.logits.squeeze(-1)
        margin = chosen_scores - rejected_scores
        loss = -F.logsigmoid(margin).mean()
        if return_outputs:
            return loss, {"chosen_scores": chosen_scores, "rejected_scores": rejected_scores, "margin": margin}
        return loss


@torch.no_grad()
def compute_train_metrics(model, tokenizer, train_ds: Dataset, batch_size=8):
    model.eval()
    total, correct = 0, 0
    margins_all = []

    for start in range(0, len(train_ds), batch_size):
        end = min(start + batch_size, len(train_ds))
        batch = train_ds[start:end]
        chosen_texts = batch["chosen_text"]
        rejected_texts = batch["rejected_text"]

        c_enc = tokenizer(
            chosen_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)
        r_enc = tokenizer(
            rejected_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)

        c = model(**c_enc).logits.squeeze(-1)
        r = model(**r_enc).logits.squeeze(-1)
        m = (c - r).detach().float().cpu().numpy()

        margins_all.append(m)
        correct += int((m > 0).sum())
        total += int(m.shape[0])

    margins = np.concatenate(margins_all, axis=0)
    return {
        "train_pairwise_acc": float(correct / max(1, total)),
        "train_mean_margin": float(margins.mean()),
        "train_std_margin": float(margins.std()),
    }



# Synthetic sampling NO AUG

def sample_synthetic_uniform(synth_pool: pd.DataFrame, n_total: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    if n_total <= len(synth_pool):
        idx = rng.choice(len(synth_pool), size=n_total, replace=False)
    else:
        print(f"[Warn] Requesting {n_total} synthetic > pool {len(synth_pool)}. Sampling with replacement.")
        idx = rng.choice(len(synth_pool), size=n_total, replace=True)
    return synth_pool.iloc[idx].copy().reset_index(drop=True)



# Main
def main():
    if HF_TOKEN:
        login(HF_TOKEN)

    tokenizer, model = build_tokenizer_and_model()
    data_collator = make_data_collator(tokenizer)

    # Pre-build base dataset once (fixed 1k)
    base_ds_pc = df_to_prompt_conditioned_dataset(base_df, has_prompt=base_has_prompt)

    # quick sanity print
    print("\nSanity check (base sample 0 chosen/rejected head):")
    print("--- chosen_text head ---")
    print(base_ds_pc[0]["chosen_text"][:400])
    print("--- rejected_text head ---")
    print(base_ds_pc[0]["rejected_text"][:400])

    iter_metrics = []
    TMP_TRAINER_DIR = "/tmp/rm_trainer_out"  # local scratch; no checkpoints saved

    for t in range(1, T + 1):
        print("\n" + "=" * 100)
        print(f"ITER {t}/{T} | BASE={len(base_ds_pc)} | TOTAL_SYN={TOTAL_BUDGET} (uniform sampled)")
        print("=" * 100)

        # Sample EXACTLY TOTAL_BUDGET synthetic pairs (uniform)
        sampled_synth = sample_synthetic_uniform(
            synth_pool=synth_df,
            n_total=TOTAL_BUDGET,
            seed=GLOBAL_SEED + t,
        )

        # Save sampled synth for audit (Drive) — NOT a model checkpoint
        sampled_path = os.path.join(SAMPLED_DIR, f"iter{t}_sampled_synth_{TOTAL_BUDGET}.csv")
        sampled_synth.to_csv(sampled_path, index=False)
        print(f"[OK] Saved sampled synthetic CSV → {sampled_path}")

        synth_ds_pc = df_to_prompt_conditioned_dataset(sampled_synth, has_prompt=synth_has_prompt)

        # Train dataset = base + sampled synth (prompt-conditioned)
        train_t = Dataset.from_dict({
            "chosen_text": list(base_ds_pc["chosen_text"]) + list(synth_ds_pc["chosen_text"]),
            "rejected_text": list(base_ds_pc["rejected_text"]) + list(synth_ds_pc["rejected_text"]),
        })
        print(f"Train size this iter: {len(train_t)} (= {len(base_ds_pc)} + {len(synth_ds_pc)})")

        # Metrics BEFORE
        before = compute_train_metrics(model, tokenizer, train_t, batch_size=8)
        print(f"[Before {t}] acc={before['train_pairwise_acc']:.4f} | mean_margin={before['train_mean_margin']:.4f}")

        # Fresh trainer each iter (keeps model weights; avoids trainer-state weirdness)
        if os.path.exists(TMP_TRAINER_DIR):
            shutil.rmtree(TMP_TRAINER_DIR, ignore_errors=True)
        os.makedirs(TMP_TRAINER_DIR, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=TMP_TRAINER_DIR,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            learning_rate=LR,
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO,
            num_train_epochs=1,
            logging_steps=LOGGING_STEPS,
            logging_strategy="steps",
            report_to="none",
            fp16=torch.cuda.is_available(),
            remove_unused_columns=False,
            save_strategy="no",
            save_safetensors=False,
        )

        trainer = PairwiseRewardTrainer(
            model=model,
            args=training_args,
            train_dataset=train_t,
            data_collator=data_collator,
        )

        train_out = trainer.train()
        train_loss = float(train_out.metrics.get("train_loss", np.nan))

        # Metrics AFTER
        after = compute_train_metrics(trainer.model, tokenizer, train_t, batch_size=8)
        print(f"[After  {t}] acc={after['train_pairwise_acc']:.4f} | mean_margin={after['train_mean_margin']:.4f}")

        iter_metrics.append({
            "iter": t,
            "base_size": len(base_ds_pc),
            "synthetic_sampled": len(synth_ds_pc),
            "train_size": len(train_t),
            "train_loss": train_loss,
            "acc_before": before["train_pairwise_acc"],
            "acc_after": after["train_pairwise_acc"],
            "mean_margin_before": before["train_mean_margin"],
            "mean_margin_after": after["train_mean_margin"],
            "std_margin_after": after["train_std_margin"],
        })

        print(f"[Iter {t}] train_loss={train_loss:.6f}")

        # Carry forward updated weights
        model = trainer.model
        del trainer
        torch.cuda.empty_cache()

        if os.path.exists(TMP_TRAINER_DIR):
            shutil.rmtree(TMP_TRAINER_DIR, ignore_errors=True)

    # Save metrics (Drive)
    metrics_df = pd.DataFrame(iter_metrics)
    metrics_df.to_csv(METRICS_CSV, index=False)
    print(f"\n[OK] Saved iter metrics CSV → {METRICS_CSV}")

    # Plots
    plt.figure()
    plt.plot(metrics_df["iter"], metrics_df["train_loss"], marker="o")
    plt.xlabel("Iteration")
    plt.ylabel("Train loss (Trainer train_loss)")
    plt.title("Baseline: fixed1k + sampled synthetic — train loss over iterations")
    plt.grid(True, linewidth=0.3)
    plt.tight_layout()
    plt.savefig(LOSS_PNG, dpi=200)
    plt.close()
    print(f"[OK] Saved loss plot → {LOSS_PNG}")

    plt.figure()
    plt.plot(metrics_df["iter"], metrics_df["acc_after"], marker="o")
    plt.xlabel("Iteration")
    plt.ylabel("Train Pairwise Accuracy (after)")
    plt.title("Baseline: train pairwise accuracy over iterations")
    plt.grid(True, linewidth=0.3)
    plt.tight_layout()
    plt.savefig(ACC_PNG, dpi=200)
    plt.close()
    print(f"[OK] Saved acc plot → {ACC_PNG}")

    plt.figure()
    plt.plot(metrics_df["iter"], metrics_df["mean_margin_after"], marker="o")
    plt.xlabel("Iteration")
    plt.ylabel("Train Mean Margin (after)")
    plt.title("Baseline: train mean margin over iterations")
    plt.grid(True, linewidth=0.3)
    plt.tight_layout()
    plt.savefig(MARGIN_PNG, dpi=200)
    plt.close()
    print(f"[OK] Saved margin plot → {MARGIN_PNG}")

    # FINAL PUSH ONLY
    if HF_TOKEN and HF_REPO_ID:
        print("\nPushing FINAL model + tokenizer to Hugging Face Hub (single push after training)...")
        login(token=HF_TOKEN)
        safe_push_to_hub(
            model=model,
            tokenizer=tokenizer,
            repo_id=HF_REPO_ID,
            token=HF_TOKEN,
            commit_message=f"final: baseline fixed1k + sampled synth (T={T}, B={TOTAL_BUDGET}, lr={LR}, gas={GRAD_ACCUM_STEPS})",
        )
        print(f"[OK] Final push complete → {HF_REPO_ID}")
    else:
        print("\n[Info] HF_TOKEN or HF_REPO_ID missing -> skipping HF push.")

    print("\nCOMPLETED!")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python
"""
WoN (West-of-N) Training: Top-K HIGH-MARGIN synthetic selection using current reward model

Requirements:
  pip install -q torch transformers datasets accelerate huggingface_hub pandas numpy tqdm matplotlib
"""

import os
import re
import time
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import shutil
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from huggingface_hub import login
from google.colab import drive


drive.mount("/content/drive")
HF_TOKEN = "xxxxxxxxxxx"  # <-- set "" to disable push
HF_REPO_ID = "xxxxxxxxxxx"  # <-- your WoN repo

BASE_MODEL = "microsoft/deberta-v3-base"
MAX_LENGTH = 512

BASE_CSV = "xxxxxxxx location selected 1k samples saved locally xxxxxxxx"
SYNTH_CSV = "xxxxxxxx location to the synthetic samples saved locally xxxxxxxx"

NUM_BASE = 1000

TOTAL_BUDGET = 2000      # total selected synthetic each iter
T = 5                    # iterations (each = 1 epoch)
GLOBAL_SEED = 42

# Training hyperparams
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 16
LR = 2e-5
WARMUP_RATIO = 0.0
WEIGHT_DECAY = 0.0
LOGGING_STEPS = 25

# Where to save audits/metrics (Drive)
RUN_ROOT = "/content/drive/My Drive/RM/FixedDataset/HHRLHF/won_fixed1k_topk_margin"
SELECTED_DIR = os.path.join(RUN_ROOT, "selected_synth_iters")
METRICS_CSV = os.path.join(RUN_ROOT, "iter_metrics.csv")
os.makedirs(SELECTED_DIR, exist_ok=True)
os.makedirs(RUN_ROOT, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Derived
assert TOTAL_BUDGET % NUM_BASE == 0, f"TOTAL_BUDGET must be divisible by NUM_BASE ({NUM_BASE})."
K_PER_BASE = TOTAL_BUDGET // NUM_BASE

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)


_HUMAN_RE = re.compile(r"^\s*(human|user)\s*:", re.IGNORECASE)

def looks_like_full_dialogue(text: str) -> bool:
    if text is None:
        return False
    t = str(text).strip()
    if len(t) < 20:
        return False
    return bool(_HUMAN_RE.search(t)) or ("Human:" in t[:200]) or ("Assistant:" in t[:200])

def build_pair_text(prompt: str, response: str) -> str:
    r = "" if response is None else str(response)
    if looks_like_full_dialogue(r):
        return r
    p = "" if prompt is None else str(prompt)
    return f"Human: {p}\n\nAssistant: {r}"

def load_base_df():
    if not os.path.exists(BASE_CSV):
        raise FileNotFoundError(f"Missing BASE_CSV: {BASE_CSV}")
    df = pd.read_csv(BASE_CSV)
    need = {"source_global_id", "chosen", "rejected"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"BASE_CSV missing columns {miss}. Found {list(df.columns)}")
    if len(df) != NUM_BASE:
        print(f"[Warn] BASE_CSV has {len(df)} rows (expected {NUM_BASE}). Using all rows.")
    return df.reset_index(drop=True)

def load_synth_df():
    if not os.path.exists(SYNTH_CSV):
        raise FileNotFoundError(f"Missing SYNTH_CSV: {SYNTH_CSV}")
    df = pd.read_csv(SYNTH_CSV)
    need = {"source_global_id", "chosen", "rejected"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"SYNTH_CSV missing columns {miss}. Found {list(df.columns)}")

    if "pair_type" in df.columns:
        df = df[df["pair_type"] != "original"].copy()
    if "pair_id" in df.columns:
        df = df[df["pair_id"] != 0].copy()

    df = df.reset_index(drop=True)
    return df

base_df = load_base_df()
synth_df = load_synth_df()

base_has_prompt = "prompt" in base_df.columns
synth_has_prompt = "prompt" in synth_df.columns

print(f"Base pairs: {len(base_df)} | base_has_prompt={base_has_prompt}")
print(f"Synth pool (excluding originals): {len(synth_df)} | synth_has_prompt={synth_has_prompt}")
print(f"WoN selection: TOTAL_BUDGET={TOTAL_BUDGET}, NUM_BASE={NUM_BASE}, K_PER_BASE={K_PER_BASE}")


# Tokenizer + model

def build_tokenizer_and_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1)

    # ensure embeddings match if we added pad token
    if getattr(model, "resize_token_embeddings", None) is not None:
        model.resize_token_embeddings(len(tokenizer))

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.problem_type = "regression"
    model.config.use_cache = False
    model.to(DEVICE)
    return tokenizer, model


# Pairwise trainer

def make_data_collator(tokenizer):
    def collate_fn(batch):
        chosen_texts = [str(ex["chosen_text"]) for ex in batch]
        rejected_texts = [str(ex["rejected_text"]) for ex in batch]

        chosen_enc = tokenizer(
            chosen_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        )
        rejected_enc = tokenizer(
            rejected_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        )

        return {
            "chosen_input_ids": chosen_enc["input_ids"],
            "chosen_attention_mask": chosen_enc["attention_mask"],
            "rejected_input_ids": rejected_enc["input_ids"],
            "rejected_attention_mask": rejected_enc["attention_mask"],
        }
    return collate_fn


class PairwiseRewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        c_out = model(
            input_ids=inputs["chosen_input_ids"],
            attention_mask=inputs["chosen_attention_mask"],
        )
        r_out = model(
            input_ids=inputs["rejected_input_ids"],
            attention_mask=inputs["rejected_attention_mask"],
        )
        c = c_out.logits.squeeze(-1)
        r = r_out.logits.squeeze(-1)
        margin = c - r
        loss = -F.logsigmoid(margin).mean()
        return (loss, {"margin": margin}) if return_outputs else loss

def df_to_prompt_conditioned_dataset(df_: pd.DataFrame, has_prompt: bool) -> Dataset:
    prompts = df_["prompt"].astype(str).tolist() if (has_prompt and "prompt" in df_.columns) else [""] * len(df_)
    chosen_texts = [build_pair_text(prompts[i], df_.iloc[i]["chosen"]) for i in range(len(df_))]
    rejected_texts = [build_pair_text(prompts[i], df_.iloc[i]["rejected"]) for i in range(len(df_))]
    return Dataset.from_dict({"chosen_text": chosen_texts, "rejected_text": rejected_texts})



# WoN
@torch.inference_mode()
def compute_margins_for_pairs(model, tokenizer, ds_pc: Dataset, batch_size: int = 8) -> np.ndarray:
    model.eval()
    margins = []

    for start in tqdm(range(0, len(ds_pc), batch_size), desc="Scoring margins", leave=False):
        end = min(start + batch_size, len(ds_pc))
        batch = ds_pc[start:end]

        c_enc = tokenizer(
            batch["chosen_text"], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)
        r_enc = tokenizer(
            batch["rejected_text"], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)

        c = model(**c_enc).logits.squeeze(-1)
        r = model(**r_enc).logits.squeeze(-1)
        m = (c - r).detach().cpu().numpy()
        margins.append(m)

    return np.concatenate(margins, axis=0)


def won_select_topk_by_margin(
    model,
    tokenizer,
    base_df: pd.DataFrame,
    synth_df: pd.DataFrame,
    k_per_base: int,
    seed: int,
) -> pd.DataFrame:

    synth_pc = df_to_prompt_conditioned_dataset(synth_df, has_prompt=("prompt" in synth_df.columns))

    margins = compute_margins_for_pairs(model, tokenizer, synth_pc, batch_size=8)

    scored = synth_df.copy()
    scored["won_margin"] = margins.astype(float)

    rng = np.random.default_rng(seed)

    selected_parts = []
    base_sources = [int(x) for x in base_df["source_global_id"].tolist()]

    grouped = {int(gid): grp for gid, grp in scored.groupby("source_global_id")}

    for i, gid in enumerate(tqdm(base_sources, desc="Selecting top-K per base", leave=False)):
        grp = grouped.get(gid, None)
        if grp is None or len(grp) == 0:

            global_top = scored.nlargest(k_per_base, "won_margin")
            pick = global_top.copy()
        else:
            if len(grp) >= k_per_base:
                pick = grp.nlargest(k_per_base, "won_margin").copy()
            else:
                top = grp.sort_values("won_margin", ascending=False).reset_index(drop=True)
                idx = rng.choice(len(top), size=k_per_base, replace=True)
                pick = top.iloc[idx].copy()

        pick["base_row_local_id"] = i
        selected_parts.append(pick)

    selected = pd.concat(selected_parts, ignore_index=True)

    expected = len(base_df) * k_per_base
    if len(selected) != expected:
        print(f"[Warn] Selected {len(selected)} rows, expected {expected}. Fixing by trim/pad.")
        if len(selected) > expected:
            selected = selected.iloc[:expected].copy()
        else:
            pad = scored.nlargest(expected - len(selected), "won_margin").copy()
            pad["base_row_local_id"] = -1
            selected = pd.concat([selected, pad], ignore_index=True)

    keep_cols = [c for c in ["source_global_id", "chosen", "rejected", "won_margin", "base_row_local_id"] if c in selected.columns]
    selected = selected[keep_cols].reset_index(drop=True)
    return selected



# MAIN

def main():
    if HF_TOKEN:
        login(token=HF_TOKEN)

    tokenizer, model = build_tokenizer_and_model()
    data_collator = make_data_collator(tokenizer)

    # Base prompt-conditioned dataset (fixed)
    base_pc = df_to_prompt_conditioned_dataset(base_df, has_prompt=base_has_prompt)

    iter_metrics = []
    TMP_DIR = "/tmp/won_rm_trainer_out"

    for t in range(1, T + 1):
        print("\n" + "=" * 110)
        print(f"WoN ITER {t}/{T} | BASE={len(base_df)} | K_PER_BASE={K_PER_BASE} | TOTAL_BUDGET={TOTAL_BUDGET}")
        print("=" * 110)

        selected_synth = won_select_topk_by_margin(
            model=model,
            tokenizer=tokenizer,
            base_df=base_df,
            synth_df=synth_df,
            k_per_base=K_PER_BASE,
            seed=GLOBAL_SEED + 1000 * t,
        )


        selected_path = os.path.join(SELECTED_DIR, f"iter{t}_won_selected_topK{K_PER_BASE}_B{TOTAL_BUDGET}.csv")
        selected_synth.to_csv(selected_path, index=False)
        print(f"[OK] Saved selected synth audit CSV → {selected_path}")
        selected_pc = df_to_prompt_conditioned_dataset(selected_synth, has_prompt=("prompt" in selected_synth.columns))


        train_t = Dataset.from_dict({
            "chosen_text": list(base_pc["chosen_text"]) + list(selected_pc["chosen_text"]),
            "rejected_text": list(base_pc["rejected_text"]) + list(selected_pc["rejected_text"]),
        })
        print(f"Train size this iter: {len(train_t)} (= base {len(base_pc)} + selected {len(selected_pc)})")

        if os.path.exists(TMP_DIR):
            shutil.rmtree(TMP_DIR, ignore_errors=True)
        os.makedirs(TMP_DIR, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=TMP_DIR,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            learning_rate=LR,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            num_train_epochs=1,
            logging_steps=LOGGING_STEPS,
            logging_strategy="steps",
            report_to="none",
            fp16=torch.cuda.is_available(),
            remove_unused_columns=False,
            save_strategy="no",
            save_safetensors=False,
            seed=GLOBAL_SEED + t,
        )

        trainer = PairwiseRewardTrainer(
            model=model,
            args=training_args,
            train_dataset=train_t,
            data_collator=data_collator,
        )

        out = trainer.train()
        train_loss = float(out.metrics.get("train_loss", np.nan))

        mean_sel_margin = float(selected_synth["won_margin"].mean()) if "won_margin" in selected_synth.columns else float("nan")

        print(f"[Iter {t}] train_loss={train_loss:.6f} | mean_selected_margin={mean_sel_margin:.6f}")

        iter_metrics.append({
            "iter": t,
            "base_size": len(base_df),
            "selected_synth": len(selected_synth),
            "train_size": len(train_t),
            "train_loss": train_loss,
            "TOTAL_BUDGET": TOTAL_BUDGET,
            "K_PER_BASE": K_PER_BASE,
            "mean_selected_margin": mean_sel_margin,
        })

        model = trainer.model
        del trainer
        torch.cuda.empty_cache()

        if os.path.exists(TMP_DIR):
            shutil.rmtree(TMP_DIR, ignore_errors=True)

    # Save metrics
    pd.DataFrame(iter_metrics).to_csv(METRICS_CSV, index=False)
    print(f"\n[OK] Saved metrics CSV → {METRICS_CSV}")

    metrics_df = pd.DataFrame(iter_metrics).sort_values("iter")
    LOSS_PNG = os.path.join(RUN_ROOT, "mean_train_loss_over_iters.png")

    if len(metrics_df) > 1 and "train_loss" in metrics_df.columns:
        plt.figure()
        plt.plot(metrics_df["iter"], metrics_df["train_loss"], marker="o")
        plt.xlabel("Iteration")
        plt.ylabel("Train loss (Trainer train_loss)")
        plt.title("WoN: train loss over iterations")
        plt.grid(True, linewidth=0.3)
        plt.tight_layout()
        plt.savefig(LOSS_PNG, dpi=200)
        plt.close()
        print(f"[OK] Saved loss curve → {LOSS_PNG}")

    # FINAL PUSH
    if HF_TOKEN and HF_REPO_ID:
        print("\nPushing FINAL model + tokenizer to Hugging Face Hub...")
        model.push_to_hub(
            HF_REPO_ID,
            token=HF_TOKEN,
            commit_message=f"final: WoN fixed1k topK={K_PER_BASE} (T={T}, B={TOTAL_BUDGET}, lr={LR}, gas={GRAD_ACCUM_STEPS})",
            max_shard_size="200MB",
        )
        tokenizer.push_to_hub(
            HF_REPO_ID,
            token=HF_TOKEN,
            commit_message="final: tokenizer (WoN)",
        )
        print(f"[OK] Final push complete → {HF_REPO_ID}")
    else:
        print("\n[Info] HF_TOKEN or HF_REPO_ID missing -> skipping HF push.")

    print("\nCOMPLETED!")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python
"""
✅ CORRECTED MARS TRAINING (Fixed 1k base + margin-adaptive *prompt-conditioned* synthetic sampling)

What this script does:
- Base = fixed 1k pairs from:
    /content/drive/My Drive/RM/FixedDataset/HHRLHF/dataset/hh_rlhf_1k.csv
- Synthetic pool = paraphrased pairs from:
    /content/drive/My Drive/RM/FixedDataset/HHRLHF/synthetic/hh_rlhf_1k_aug26.csv
  (excludes "original" rows so pool is truly synthetic)

MARS per iteration t=1..T:
  1) Compute margins m_i = r(x_i, y_i+) - r(x_i, y_i-) on the FIXED 1k base pairs
     ✅ uses prompt-conditioning if a "prompt" column exists OR if chosen already contains full HH dialogue
  2) q_i = softmax(-alpha * |m_i|)  (focus budget on ambiguous/small |m_i|)
  3) Convert to integer budgets b_i summing to TOTAL_BUDGET (largest remainder)
  4) For each base sample i, sample exactly b_i synthetic pairs restricted to source_global_id
     - if that source has fewer synthetic rows, sample with replacement
     - if that source has none, fall back to global synthetic pool
  5) Train 1 epoch on: base_1k + sampled_synth_this_iter  (size = 1000 + TOTAL_BUDGET)

✅ NO local/Drive model checkpoints:
  - save_strategy="no"
  - save_safetensors=False
  - no save_model() calls

✅ Pushes to HF Hub ONLY ONCE AFTER ALL TRAINING COMPLETES (final push)

Saves to Drive ONLY:
- sampled synthetic CSV per iteration (audit)
- metrics CSV + simple plots

Requirements:
  pip install -q torch transformers datasets accelerate huggingface_hub pandas numpy tqdm requests matplotlib
"""

import os
import re
import time
import random
import shutil
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from huggingface_hub import login
from huggingface_hub.utils import configure_http_backend


# =========================
# CONFIG
# =========================
HF_TOKEN = "xxxxxxxxxxxx"  # <-- your hf_... token ("" disables pushes)
HF_REPO_ID = "xxxxxxxxxxx"  # <-- set your repo name

BASE_MODEL = "microsoft/deberta-v3-base"
MAX_LENGTH = 512

BASE_CSV = "xxxxxxxx location selected 1k samples saved locally xxxxxxxx"
SYNTH_CSV = "xxxxxxxx location selected synthetic samples saved locally xxxxxxxx"

NUM_BASE = 1000

# MARS params
TOTAL_BUDGET = 2000
ALPHA = 0.1 #defined as tau in the paper
T = 5

# Training hyperparams
PER_DEVICE_TRAIN_BATCH = 4
GRAD_ACCUM_STEPS = 16
LR = 2e-5
WARMUP_RATIO = 0.0
WEIGHT_DECAY = 0.0
LOGGING_STEPS = 5

# Save ONLY audit + metrics to Drive (not checkpoints)
RUN_ROOT = "/content/drive/My Drive/RM/FixedDataset/HHRLHF/mars_fixed1k_sampledsynth_promptcond"
SAMPLED_DIR = os.path.join(RUN_ROOT, "sampled_synth_iters")
METRICS_CSV = os.path.join(RUN_ROOT, "iter_metrics.csv")
LOSS_PNG = os.path.join(RUN_ROOT, "train_loss_over_iters.png")
ACC_PNG = os.path.join(RUN_ROOT, "train_pairwise_acc_over_iters.png")
MARGIN_PNG = os.path.join(RUN_ROOT, "train_mean_margin_over_iters.png")
os.makedirs(SAMPLED_DIR, exist_ok=True)
os.makedirs(RUN_ROOT, exist_ok=True)

GLOBAL_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# HF timeouts
os.environ.setdefault("HF_HUB_HTTP_TIMEOUT", "180")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "180")

def _hf_backend_factory() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=8,
        connect=8,
        read=8,
        status=8,
        backoff_factor=1.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["HEAD", "GET", "POST", "PUT", "DELETE", "PATCH"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=16, pool_maxsize=16)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

configure_http_backend(backend_factory=_hf_backend_factory)


def safe_push_to_hub(model, tokenizer, repo_id: str, token: str, commit_message: str, max_retries: int = 6):
    if not token or not repo_id:
        return
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            print(f"[HF] Push attempt {attempt}/{max_retries}: {repo_id} | {commit_message}")
            model.push_to_hub(
                repo_id,
                token=token,
                commit_message=commit_message,
                max_shard_size="200MB",
            )
            tokenizer.push_to_hub(
                repo_id,
                token=token,
                commit_message=commit_message,
            )
            print("[HF] Push complete.")
            return
        except Exception as e:
            last_err = e
            wait = min(60, 2 ** attempt)
            print(f"[HF] Push failed ({type(e).__name__}: {e}). Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError(f"HF push failed after {max_retries} attempts. Last error: {last_err}")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)


_HUMAN_RE = re.compile(r"^\s*(human|user)\s*:", re.IGNORECASE)

def looks_like_full_dialogue(text: str) -> bool:
    if text is None:
        return False
    t = str(text).strip()
    if len(t) < 20:
        return False
    return bool(_HUMAN_RE.search(t)) or ("Human:" in t[:200]) or ("Assistant:" in t[:200])

def build_pair_text(prompt: str, response: str) -> str:
    r = "" if response is None else str(response)
    if looks_like_full_dialogue(r):
        return r
    p = "" if prompt is None else str(prompt)
    return f"Human: {p}\n\nAssistant: {r}"


def load_base_df():
    if not os.path.exists(BASE_CSV):
        raise FileNotFoundError(f"Missing BASE_CSV: {BASE_CSV}")
    df = pd.read_csv(BASE_CSV)
    need = {"source_global_id", "chosen", "rejected"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"BASE_CSV missing columns {miss}. Found {list(df.columns)}")
    if len(df) != NUM_BASE:
        print(f"[Warn] BASE_CSV has {len(df)} rows (expected {NUM_BASE}). Using all rows.")
    return df.reset_index(drop=True)


def load_synth_df():
    if not os.path.exists(SYNTH_CSV):
        raise FileNotFoundError(f"Missing SYNTH_CSV: {SYNTH_CSV}")
    df = pd.read_csv(SYNTH_CSV)
    need = {"source_global_id", "chosen", "rejected"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"SYNTH_CSV missing columns {miss}. Found {list(df.columns)}")

    # Exclude originals to ensure "synthetic" pool
    if "pair_type" in df.columns:
        df = df[df["pair_type"] != "original"].copy()
    if "pair_id" in df.columns:
        df = df[df["pair_id"] != 0].copy()

    return df.reset_index(drop=True)


base_df = load_base_df()
synth_df = load_synth_df()

base_has_prompt = "prompt" in base_df.columns
synth_has_prompt = "prompt" in synth_df.columns

print(f"Base pairs: {len(base_df)} | base_has_prompt={base_has_prompt}")
print(f"Synthetic pool (excluding originals): {len(synth_df)} | synth_has_prompt={synth_has_prompt}")

# Group synthetic by source_global_id for MARS per-sample sampling
synth_by_source = {int(gid): grp.reset_index(drop=True) for gid, grp in synth_df.groupby("source_global_id")}
missing_sources = [int(g) for g in base_df["source_global_id"].unique() if int(g) not in synth_by_source]
if missing_sources:
    print(f"[Warn] {len(missing_sources)} base sources have no synthetic rows. Will fall back to GLOBAL pool for them.")


def build_tokenizer_and_model():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

    # safer pad handling
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=1)

    if getattr(model, "resize_token_embeddings", None) is not None:
        model.resize_token_embeddings(len(tokenizer))

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.problem_type = "regression"
    model.config.use_cache = False
    model.to(DEVICE)
    return tokenizer, model


def make_data_collator(tokenizer):
    def collate_fn(batch):
        chosen_texts = [str(ex["chosen_text"]) for ex in batch]
        rejected_texts = [str(ex["rejected_text"]) for ex in batch]

        chosen_enc = tokenizer(
            chosen_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        rejected_enc = tokenizer(
            rejected_texts, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        )

        return {
            "chosen_input_ids": chosen_enc["input_ids"],
            "chosen_attention_mask": chosen_enc["attention_mask"],
            "rejected_input_ids": rejected_enc["input_ids"],
            "rejected_attention_mask": rejected_enc["attention_mask"],
        }
    return collate_fn


class PairwiseRewardTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        chosen_out = model(
            input_ids=inputs["chosen_input_ids"],
            attention_mask=inputs["chosen_attention_mask"],
        )
        rejected_out = model(
            input_ids=inputs["rejected_input_ids"],
            attention_mask=inputs["rejected_attention_mask"],
        )
        chosen_scores = chosen_out.logits.squeeze(-1)
        rejected_scores = rejected_out.logits.squeeze(-1)
        margin = chosen_scores - rejected_scores
        loss = -F.logsigmoid(margin).mean()
        if return_outputs:
            return loss, {"chosen_scores": chosen_scores, "rejected_scores": rejected_scores, "margin": margin}
        return loss

def df_to_prompt_conditioned_dataset(df_: pd.DataFrame, has_prompt: bool) -> Dataset:
    prompts = df_["prompt"].astype(str).tolist() if (has_prompt and "prompt" in df_.columns) else [""] * len(df_)
    chosen_texts = [build_pair_text(prompts[i], df_.iloc[i]["chosen"]) for i in range(len(df_))]
    rejected_texts = [build_pair_text(prompts[i], df_.iloc[i]["rejected"]) for i in range(len(df_))]
    return Dataset.from_dict({"chosen_text": chosen_texts, "rejected_text": rejected_texts})


# Margins and budgets

@torch.inference_mode()
def compute_margins_on_base(model, tokenizer, base_pc_ds: Dataset, batch_size: int = 8) -> np.ndarray:
    model.eval()
    margins = []

    for start in range(0, len(base_pc_ds), batch_size):
        end = min(start + batch_size, len(base_pc_ds))
        batch = base_pc_ds[start:end]
        chosen_texts = batch["chosen_text"]
        rejected_texts = batch["rejected_text"]

        chosen_enc = tokenizer(
            chosen_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)
        rejected_enc = tokenizer(
            rejected_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)

        rc = model(**chosen_enc).logits.squeeze(-1)
        rr = model(**rejected_enc).logits.squeeze(-1)
        m = (rc - rr).detach().cpu().numpy()
        margins.append(m)

    return np.concatenate(margins, axis=0)


def _softmax(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / (np.sum(ex) + 1e-12)


def compute_integer_budgets_from_margins(margins: np.ndarray, alpha: float, total_budget: int, seed: int):
    """
    q_i = softmax(-alpha * |margin_i|)
    """
    # q_i
    scores = -alpha * np.abs(margins)
    q = _softmax(scores)

    # raw_b_i
    raw = total_budget * q

    # largest remainder rounding
    b_floor = np.floor(raw).astype(int)
    remainder = raw - b_floor
    short = int(total_budget - b_floor.sum())

    rng = np.random.default_rng(seed)
    order = np.lexsort((rng.random(len(remainder)), -remainder))  # remainder desc, random tie-break

    b = b_floor.copy()
    if short > 0:
        b[order[:short]] += 1
    elif short < 0:
        b[order[short:]] = np.maximum(0, b[order[short:]] - 1)

    diff = int(total_budget - b.sum())
    if diff != 0:
        if diff > 0:
            b[order[:diff]] += 1
        else:
            b[order[diff:]] = np.maximum(0, b[order[diff:]] - 1)

    if int(b.sum()) != int(total_budget):
        raise RuntimeError(f"Budget sum mismatch: {b.sum()} vs {total_budget}")

    return b, q, raw


# MARS synthetic sampling

def sample_rows_from_group(grp: pd.DataFrame, n: int, rng: np.random.Generator) -> pd.DataFrame:
    if n <= 0 or len(grp) == 0:
        return grp.iloc[0:0].copy()
    replace = n > len(grp)
    idx = rng.choice(len(grp), size=n, replace=replace)
    return grp.iloc[idx].copy()


def build_mars_sampled_synth(base_df: pd.DataFrame, budgets: np.ndarray, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    sampled_parts = []
    global_pool = synth_df

    for i in range(len(base_df)):
        b_i = int(budgets[i])
        if b_i <= 0:
            continue

        gid = int(base_df.loc[i, "source_global_id"])
        grp = synth_by_source.get(gid, None)
        if grp is None or len(grp) == 0:
            grp = global_pool

        part = sample_rows_from_group(grp, b_i, rng)
        if len(part) == 0:
            continue

        part = part[["source_global_id", "chosen", "rejected"]].copy()
        part["base_row_local_id"] = i
        part["b_i"] = b_i
        sampled_parts.append(part)

    out = pd.concat(sampled_parts, ignore_index=True) if sampled_parts else pd.DataFrame(
        columns=["source_global_id", "chosen", "rejected", "base_row_local_id", "b_i"]
    )
    return out


@torch.no_grad()
def compute_train_metrics(model, tokenizer, train_pc_ds: Dataset, batch_size=8):
    model.eval()
    total, correct = 0, 0
    margins_all = []

    for start in range(0, len(train_pc_ds), batch_size):
        end = min(start + batch_size, len(train_pc_ds))
        batch = train_pc_ds[start:end]

        c_enc = tokenizer(
            batch["chosen_text"], padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)
        r_enc = tokenizer(
            batch["rejected_text"], padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        ).to(DEVICE)

        c = model(**c_enc).logits.squeeze(-1)
        r = model(**r_enc).logits.squeeze(-1)
        m = (c - r).detach().float().cpu().numpy()

        margins_all.append(m)
        correct += int((m > 0).sum())
        total += int(m.shape[0])

    margins = np.concatenate(margins_all, axis=0)
    return {
        "train_pairwise_acc": float(correct / max(1, total)),
        "train_mean_margin": float(margins.mean()),
        "train_std_margin": float(margins.std()),
    }


# MAIN

def main():
    if HF_TOKEN:
        login(HF_TOKEN)

    tokenizer, model = build_tokenizer_and_model()
    data_collator = make_data_collator(tokenizer)

    base_pc_ds = df_to_prompt_conditioned_dataset(base_df, has_prompt=base_has_prompt)

    print("\nSanity check (base sample 0 chosen/rejected head):")
    print("--- chosen_text head ---")
    print(base_pc_ds[0]["chosen_text"][:400])
    print("--- rejected_text head ---")
    print(base_pc_ds[0]["rejected_text"][:400])

    iter_metrics = []
    TMP_TRAINER_DIR = "/tmp/mars_rm_trainer_out"

    for t in range(1, T + 1):
        print("\n" + "=" * 110)
        print(f"MARS ITER {t}/{T} | BASE={len(base_df)} | B={TOTAL_BUDGET} | alpha={ALPHA}")
        print("=" * 110)
        margins = compute_margins_on_base(model, tokenizer, base_pc_ds, batch_size=8)

        budgets, q, raw = compute_integer_budgets_from_margins(
            margins=margins,
            alpha=ALPHA,
            total_budget=TOTAL_BUDGET,
            seed=GLOBAL_SEED + 1000 * t,
        )

        print("xxxxxxxxx Budget summary xxxxxxxxxx")
        print(f"sum(b_i)={int(budgets.sum())} | min={int(budgets.min())} | max={int(budgets.max())} | mean={budgets.mean():.4f}")

        sampled_synth = build_mars_sampled_synth(
            base_df=base_df,
            budgets=budgets,
            seed=GLOBAL_SEED + 2000 * t,
        )

        if len(sampled_synth) != TOTAL_BUDGET:
            print(f"[Warn] sampled_synth rows = {len(sampled_synth)} but expected B={TOTAL_BUDGET}. "
                  f"This usually means synth pool is missing for some sources AND global pool is unexpectedly small.")
        else:
            print(f"[OK] Sampled exactly B={TOTAL_BUDGET} synthetic pairs this iter.")
        sampled_path = os.path.join(SAMPLED_DIR, f"iter{t}_mars_sampled_synth_B{TOTAL_BUDGET}_alpha{ALPHA}.csv")
        sampled_synth.to_csv(sampled_path, index=False)
        print(f"[OK] Saved sampled synth audit CSV → {sampled_path}")

        synth_pc_ds = df_to_prompt_conditioned_dataset(sampled_synth, has_prompt=synth_has_prompt)

        train_t = Dataset.from_dict({
            "chosen_text": list(base_pc_ds["chosen_text"]) + list(synth_pc_ds["chosen_text"]),
            "rejected_text": list(base_pc_ds["rejected_text"]) + list(synth_pc_ds["rejected_text"]),
        })
        print(f"Train size this iter: {len(train_t)} (= base {len(base_pc_ds)} + synth {len(synth_pc_ds)})")

        before = compute_train_metrics(model, tokenizer, train_t, batch_size=8)
        print(f"[Before {t}] acc={before['train_pairwise_acc']:.4f} | mean_margin={before['train_mean_margin']:.4f}")
        if os.path.exists(TMP_TRAINER_DIR):
            shutil.rmtree(TMP_TRAINER_DIR, ignore_errors=True)
        os.makedirs(TMP_TRAINER_DIR, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=TMP_TRAINER_DIR,
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            learning_rate=LR,
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO,
            num_train_epochs=1,
            logging_steps=LOGGING_STEPS,
            logging_strategy="steps",
            report_to="none",
            fp16=torch.cuda.is_available(),
            remove_unused_columns=False,
            save_strategy="no",
            save_safetensors=False,
            seed=GLOBAL_SEED + t,
        )

        trainer = PairwiseRewardTrainer(
            model=model,
            args=training_args,
            train_dataset=train_t,
            data_collator=data_collator,
        )

        train_out = trainer.train()
        train_loss = float(train_out.metrics.get("train_loss", np.nan))
        print(f"[Iter {t}] train_loss={train_loss:.6f}")
        after = compute_train_metrics(trainer.model, tokenizer, train_t, batch_size=8)
        print(f"[After  {t}] acc={after['train_pairwise_acc']:.4f} | mean_margin={after['train_mean_margin']:.4f}")

        iter_metrics.append({
            "iter": t,
            "base_size": len(base_pc_ds),
            "synthetic_sampled": len(synth_pc_ds),
            "train_size": len(train_t),
            "train_loss": train_loss,
            "B": TOTAL_BUDGET,
            "alpha": ALPHA,
            "sum_b": int(budgets.sum()),
            "min_b": int(budgets.min()),
            "max_b": int(budgets.max()),
            "mean_abs_margin_base": float(np.mean(np.abs(margins))),
            "median_abs_margin_base": float(np.median(np.abs(margins))),
            "acc_before": before["train_pairwise_acc"],
            "acc_after": after["train_pairwise_acc"],
            "mean_margin_before": before["train_mean_margin"],
            "mean_margin_after": after["train_mean_margin"],
            "std_margin_after": after["train_std_margin"],
        })

        model = trainer.model
        del trainer
        torch.cuda.empty_cache()

        if os.path.exists(TMP_TRAINER_DIR):
            shutil.rmtree(TMP_TRAINER_DIR, ignore_errors=True)

    metrics_df = pd.DataFrame(iter_metrics)
    metrics_df.to_csv(METRICS_CSV, index=False)
    print(f"\n[OK] Saved metrics CSV → {METRICS_CSV}")

    plt.figure()
    plt.plot(metrics_df["iter"], metrics_df["train_loss"], marker="o")
    plt.xlabel("Iteration")
    plt.ylabel("Train loss (Trainer train_loss)")
    plt.title("MARS: train loss over iterations")
    plt.grid(True, linewidth=0.3)
    plt.tight_layout()
    plt.savefig(LOSS_PNG, dpi=200)
    plt.close()
    print(f"[OK] Saved loss plot → {LOSS_PNG}")

    plt.figure()
    plt.plot(metrics_df["iter"], metrics_df["acc_after"], marker="o")
    plt.xlabel("Iteration")
    plt.ylabel("Train Pairwise Accuracy (after)")
    plt.title("MARS: train pairwise accuracy over iterations")
    plt.grid(True, linewidth=0.3)
    plt.tight_layout()
    plt.savefig(ACC_PNG, dpi=200)
    plt.close()
    print(f"[OK] Saved acc plot → {ACC_PNG}")

    plt.figure()
    plt.plot(metrics_df["iter"], metrics_df["mean_margin_after"], marker="o")
    plt.xlabel("Iteration")
    plt.ylabel("Train Mean Margin (after)")
    plt.title("MARS: train mean margin over iterations")
    plt.grid(True, linewidth=0.3)
    plt.tight_layout()
    plt.savefig(MARGIN_PNG, dpi=200)
    plt.close()
    print(f"[OK] Saved margin plot → {MARGIN_PNG}")

    if HF_TOKEN and HF_REPO_ID:
        print("\nPushing FINAL model + tokenizer to Hugging Face Hub (single push after training)...")
        login(token=HF_TOKEN)
        safe_push_to_hub(
            model=model,
            tokenizer=tokenizer,
            repo_id=HF_REPO_ID,
            token=HF_TOKEN,
            commit_message=f"final: MARS fixed1k (T={T}, alpha={ALPHA}, B={TOTAL_BUDGET}, lr={LR}, gas={GRAD_ACCUM_STEPS})",
        )
        print(f"[OK] Final push complete → {HF_REPO_ID}")
    else:
        print("\n[Info] HF_TOKEN or HF_REPO_ID missing -> skipping HF push.")

    print("\nCOMPLETED!")


if __name__ == "__main__":
    main()
